In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd

ddir = 'data/'

df = pd.read_csv(os.path.join(ddir,'df_samp_20to30_10000.csv'))
df.head(5)

,smiles
0,COc1cccc2oc(=O)cc(N3CCOCC3)c12
1,CNCCCCNC1(c2cc3ccccc3s2)CCCCC1
2,NCC1OC2OCCC1(O)C2O
3,Cc1c(O)c(CN)c(Cl)c(C)c1Cl
4,CCCC1C2CNCC12c1ccc(Cl)c(Cl)c1


In [2]:
from rdkit.Chem import PandasTools
def displaydf(df):
    return HTML(df.to_html(notebook=True))

PandasTools.AddMoleculeColumnToFrame(df,'smiles','mol',includeFingerprints=False)

from data_utils import *
atom_to_cnt = get_atom_cnts(df.smiles)
# atom_to_cnt

In [3]:
from rdkit_utils import *
from graph_augs import *

from rdkit import RDLogger  
RDLogger.DisableLog('rdApp.*')

def get_augs(smiles,maximum=10):
    
    mol = Chem.MolFromSmiles(smiles)

    idc = [i for i in range(0,(mol.GetNumAtoms()))]
    random.shuffle(idc)
    
    goods = 0 
    aug_smiles = []
    for i in idc:
        if goods==maximum:
            break
            
        atom_type = get_weighted_random_atom(atom_to_cnt)
        
        try: 
            mol_aug = add_atom_to_mol(mol, atom_type, i, clean_aroms = True)
            if mol_aug.GetNumAtoms()==0:
                continue
            else:
                sm = Chem.MolToSmiles(mol_aug)
                goods+=1
                aug_smiles.append(sm)
        except Exception as e:
            continue
            
    return (smiles, aug_smiles)

from joblib import Parallel, delayed
import multiprocessing

parallelizer = Parallel(n_jobs=multiprocessing.cpu_count()-1, backend= 'multiprocessing' )
augs_tasks = (delayed(get_augs)(smiles) for smiles in df.smiles)
smiles_to_augs = parallelizer(augs_tasks)

smiles_to_augs = {k:v for k,v in smiles_to_augs}

In [5]:
from rdkit_utils import *
from graph_augs import *

no_augs = [k for k,v in smiles_to_augs.items() if len(v)==0]
# print(len(no_augs))

# for smiles in no_augs:
#     print('= = = = = = = =')
#     mol = Chem.MolFromSmiles(smiles)
#     mol_show = show_atom_index(mol)
#     display(Chem.MolToSmiles(mol), mol)
#     try:
#         mol_new = add_atom_to_mol(mol,'C',3)
#         display(mol_new)
#     except:
#         continue

In [6]:
for k,v in smiles_to_augs.items():
    print(k,len(v))

COc1cccc2oc(=O)cc(N3CCOCC3)c12 9
CNCCCCNC1(c2cc3ccccc3s2)CCCCC1 10
NCC1OC2OCCC1(O)C2O 9
Cc1c(O)c(CN)c(Cl)c(C)c1Cl 5
CCCC1C2CNCC12c1ccc(Cl)c(Cl)c1 10
Nc1ccc(-c2nc3cc(F)ccc3s2)cc1I 9
CCOc1ccc(C(=O)NC2=NCCS2)cc1 10
NCCc1n[nH]c(=O)n1-c1cccc(Cl)c1Cl 6
CC(=O)N(CC(=O)N(C)C)c1ccccc1 9
CC1NC(=O)C(=O)OC1C 5
CCCCCCN1CC2CN(C)CC(C1)C2(CC)CC 10
c1ccc(-c2nc(-c3nnn[nH]3)cs2)cc1 7
COC(=O)C(Cc1ccccc1)NC(=O)CN 10
CCc1nc(CC(=O)NC(C#N)CC)cs1 9
CCCCOc1cc(C(N)=O)nn1-c1ccccc1 10
CCC1=NC(c2ccccc2)N(O)C1(C)C 10
CCOC(=O)Nc1cc2ccccc2oc1=O 8
Cc1cc(O)cc(C)c1CC(N)C(=O)NCN 10
CC(=NC1CC2CCC1C2)Nc1ccccc1C 10
N#Cc1ccccc1OCCOc1ccccc1C#N 10
COCCNc1nc(NC(C)C)nc2ccsc12 10
O=c1cc(O)sc(-c2ccc(Cl)cc2)n1 7
O=C(C[n+]1ccccn1)c1ccc(Cl)cc1 9
CNc1cncc(C(N)=O)n1 5
O=c1[nH]c(N2CCOCC2)cn1-c1ccccc1 10
O=c1c2nncn2ccn1-c1ccc(F)c(F)c1 6
CC1OC2OC1C=Cc1cccc(O)c12 10
COCC(=O)N1CCC(Oc2cccc(C)c2)CC1 10
CCCC#CC#CC=CC(=O)OC 6
NC1CC(=O)c2csc(Br)c21 5
CCCCN1CC2CC(C1)CN(Cc1ccccc1)C2 10
CCCCn1c(=O)c2nc[nH]c2n(CC)c1=O 7
c1ccc(CNc2ccc(C3CCCCC3)cc2)cc1 

CS(=O)(=O)c1nc2ccccc2nc1NCCO 9
COc1cccc2cc(C(C)=O)c(=S)oc12 7
CCCCCNC1=NCCN1OCc1ccccc1OC 10
NCC1(c2cc(O)c(O)cc2N)CCCC1 10
COc1ccccc1C(=O)Nc1ccccc1SC 10
O=C(CCC1COc2ccccc2O1)NC1CCCCC1 10
Cc1ccc2c(c1)CN(c1ccccc1Cl)CO2 10
NCCCCCCCCCc1ccccn1 10
CCOC(=O)C1=CN(Cc2ccccc2)CCC1=O 10
S=c1[nH]nc(-c2ccccc2)c2ccccc12 10
CN(C)C(=O)c1cc(Br)ccc1Cl 5
NC(=S)NCCCC(N)C(=O)O 9
CC1CCCC(=NNC2=NC(=O)CS2)C1 9
NS(=O)(=O)c1ccc(SCc2ccccc2)nc1 10
S=C(NCCN1CCOCC1)Nc1ccc(Br)cc1 10
COc1ccc(C(=O)CNc2ccccc2C#N)cc1 10
O=C(O)C1CC(=O)N(C2CCCCCC2)C1 10
O=C1Nc2ccccc2C1=NNc1ccccc1F 10
O=C(NC1CCSc2ccccc21)c1ccccc1Br 10
O=C(Cc1ccccc1)Nc1nccs1 10
COC(=O)c1cc2c(cc1OC)CCCC2(C)C 9
Cc1c(C=O)cnn1CCC(=O)O 6
COC(=O)C1=C(C)NC(=O)NC1c1ccco1 8
O=S1c2ccccc2SCSc2ccccc21 10
CNC(=S)OCCC(C)CCCC(C)C 10
CCCCCCNC(=O)c1ccc(Br)o1 9
O=C1COc2cc3c(cc2N1)SCCC(=O)N3 8
CCCCCCCCC=CCCCCCCCC(=O)N(C)OC 10
CC1(C)COP(=S)(c2ccccc2)N1 10
C=CCSc1ccccc1C(=O)Nc1nc[nH]n1 10
O=C(NCCCc1ccccc1)c1ccc[nH]1 10
O=c1ccc2c(cc(O)c3ccccc32)o1 8
CCNC(C)C(=O)Nc1nsc2ccc(C)cc12 1

CC(CCc1ccccc1)NC(=O)CNCCCN(C)C 10
Oc1ccc(Cl)cc1CNn1cnnc1 8
Cc1ccccc1NC(=S)NN1CCN(C)CC1 10
O=C(Cn1cc(I)cn1)NCC1CCC1 10
CCCOCCCNC(=S)Nc1cc(Cl)ccc1OC 10
Cc1cnc(NC2=NCCC2)s1 7
Cc1cc(C)c(NC(N)=S)c(C)c1 8
Cc1ccc(N)c(NC(=O)c2cccnc2)c1 10
N#Cc1cc(C#N)c(Cl)cc1Cl 2
O=C(O)c1ccc(NC(=O)c2ccccc2)cc1 10
COC1COC2(CCN(CC3CC3)CC2)C1 10
NC1CCCCN(CC(=O)O)C1=O 8
O=c1c2ccccc2cc2n1C1CCCCC1N2 10
OCc1ccnc(-c2cc(CO)ccn2)c1 10
CCCOC(=O)C(C)=NNC(=O)c1ccco1 8
O=C(Nc1ccc([N+](=O)[O-])cn1)C1CC1 7
C=CCc1cc(OC)ccc1OCCCCn1cncn1 10
c1ccc2c(c1)Sc1ccccc1N2C1CNC1 10
CN(C)CCCCCN(C)Cc1cc2ccccc2o1 10
CC(c1noc2ccc(Cl)cc12)n1ccnc1 8
CCNC(=O)N(Cc1ccoc1)C1CCCCC1 10
Cc1cnc2c(C(=O)O)c(Cl)ccc2c1 6
COC1=CC(=O)C(C)=C2CC(C)OC=C12 7
CCNC(=O)NC(=O)CSC(C)c1ccccc1F 10
O=C(Nc1ccc2ncccc2c1)c1ccccc1 10
O=C1NC(=S)NC1=C1CCOc2ccccc21 9
CC1=CC(=O)OC2NN=C(C)C12 6
CCCCCCCCCCCCCCC(N)(CO)CO 10
NC1CC(F)(F)CC1C(=O)O 6
O=C1CCCN(c2cccc(F)c2)N1 8
Cc1ccc([N+](=O)[O-])c([N+](=O)[O-])c1 4
NC(Cc1ccccc1)c1nccs1 10
Cc1cc(C)n(C(CC(=O)O)C(=O)O)n1 7
C[N+]12CCC(C1)

O=C(O)C(CCC1CCNCC1)c1c[nH]cn1 10
c1cncc(N2CC3CNC3C2)c1 10
O=[N+]([O-])c1ccc(-c2cccnc2)s1 7
O=C1Nc2cc(CCc3ccccc3)ccc2C1=O 10
NC(CSCc1ccccc1)Cc1ccccc1 10
c1ccc2cc3ccccc3cc2c1 10
O=c1cc(-c2ccccc2)oc2ccc(O)cc12 10
COc1ccc(C2=CCCCCC2)cc1OC 10
O=C1c2cc(O)ccc2OCC1n1ccnc1 9
O=C(NN=Cc1ccncc1)c1ccc(Cl)s1 9
O=C(Cc1ccccc1)NN=Cc1ccccc1O 10
Nc1ccc(C(=O)C=Cc2ccc(I)cc2)cc1 10
CCc1cc(C(=O)Nc2cccnc2)n(C)n1 9
S=C(Nc1ccccc1)Nn1cnnc1 10
SCc1ccc(-c2ccc(CS)cc2)cc1 10
O=C(C=Cc1ccco1)Nc1ccccc1C(=O)O 10
O=C(c1ccc(Cl)cc1)N1CCSCC1 9
CN1C2CC=C(Cc3ccccc3)C1CC2 10
CC(C)CC(N)C(=O)N1CCSc2ccccc2C1 10
Cc1cccc(NS(=O)(=O)Cc2ccccc2)n1 10
C[N+]1(C)CCc2cc(O)c(O)cc2C1 9
CC=Cc1cccc2nc(OCC)oc(=O)c12 8
CN(C)c1cc(N)c2c3c(sc2n1)CCCC3 9
COC(=O)CC1(O)C=CC(=O)c2ccccc21 9
Cc1cc(N2CCN(C)CC2)nc(N)n1 8
FC1=C2C3CCC(C3)C2C1(F)F 6
Nc1ccc(N=Nc2ccccc2)c2ccccc12 10
Cc1ccccc1OCCN1CCn2c(C)nnc2C1 10
CCn1nnc(C2=CCCN(C(C)C)C2)n1 9
CC(C)NC(=S)N1CCC(c2ccccc2)=N1 10
Cn1cnnc1SCc1cc2c(cc1Cl)OCO2 7
Cc1cc(Cl)ccc1OCCCC(=O)N1CCCCC1 10
CN1CCCC1COc1cccc(C#N)c

CC1OC2(CCNCC2)CC1=O 8
CCNCCCCOc1ccc(Oc2ccccc2)cc1 10
Cc1nc2ccccc2cc1-c1cn2ccccc2n1 10
Cc1nc2c(ccc3cccnc32)c2occc12 8
COc1cccc(CNC(=O)C(C)NC(C)=O)c1 10
O=C1CSc2cccc(Cl)c2N1CC1CCCO1 10
Oc1ccc(CNc2ccccc2O)cc1 10
COc1ccccc1N1CCSC1C(N)=O 10
Cc1nc2ncnn2c(C)c1Cc1ccc(Cl)cc1 8
CSC1CN(Cc2ccccc2)C(=O)C1=O 10
O=C(O)c1ccccc1OCC1CCCO1 10
O=Nc1ccc(NOC(=O)c2ccccc2)cc1 10
O=C1NC(=O)C(=Cc2cccc(Cl)c2Cl)S1 6
CC(=O)NC1CC1c1ccccc1 10
CSc1ccc(C(=S)N2CCN(CCO)CC2)cc1 10
Cc1nn(-c2ccccc2)c(=S)[nH]1 7
O=c1c2ccccc2[nH]n1Cc1cncs1 8
FC(F)(Cl)OC(F)(F)C(F)(Cl)Cl 0
CCc1ccccc1NC(=O)C1COc2ccccc2C1 10
COc1ccc(C(=O)C=C2NCCN2)cc1 10
C=CCn1c(-c2sccc2C)n[nH]c1=S 8
CC(C)(C)NCC(O)CON=Cc1ccccc1 10
CC(=O)C(CCCCO)CCCCCCC(=O)O 10
N#CN1CCC(NC(=O)OCc2ccccc2)C1 10
NC1CC(Oc2ccccc2)c2ccccc21 10
CCCOc1cccc(C(=O)O)c1 8
c1ccc(-c2ccccc2COCC2CCCNC2)cc1 10
OC(CN1CCOCC1)CN1CCOCC1 10
O=C(NCc1ccc(F)cc1)c1cccnc1 10
C#CCSc1nncn1-c1ccc(Cl)c(Cl)c1 7
Nn1c(N2CCCCC2)nc2ccccc21 10
NC(=O)c1nc(-c2ccccc2)oc1N 7
O=C1CC23CCC4(CC2CCCC3O1)OCCO4 10
C=CCC(OC(=O)